<a href="https://colab.research.google.com/github/chaely0603-netizen/KBU202618304/blob/main/2026_10_16_%EC%9C%A4%EC%B0%AC%EC%8A%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
f = open("myfile.txt", "r", encoding="utf-8")


FileNotFoundError: [Errno 2] No such file or directory: 'myfile.txt'

In [13]:
with open("mytile.txt", "w", encoding="utf-8") as f:
  f.write("Hello, this is myfile.txt!")
print("myfile.txt 파일이 성공적으로 생성되었습니다.")

myfile.txt 파일이 성공적으로 생성되었습니다.


In [14]:
%whos

Variable   Type                          Data/Info
--------------------------------------------------
f          TextIOWrapper                 <_io.TextIOWrapper name='<...>ode='w' encoding='utf-8'>
write      builtin_function_or_method    <built-in function write>


In [20]:
with open("myfile.txt", "w", encoding="utf-8") as f:
    f.write("안녕하세요!\n")
    f.write("파이썬 공부 중입니다.")

In [22]:
with open("myfile.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(content)
print(type(content))

안녕하세요!
파이썬 공부 중입니다.
<class 'str'>


In [23]:
import io

f = io.StringIO("첫 번째 줄\n두 번째 줄")

print(f.read())

첫 번째 줄
두 번째 줄


In [25]:
!pip install faker

import csv
import sqlite3
from faker import Faker

# ----------------------------------------
# 1. Faker로 CSV 파일 생성
# ----------------------------------------
fake = Faker("ko_KR")

CSV_FILE = "users.csv"
DB_FILE = "users.db"

def create_csv(filename=CSV_FILE, count=100):
    """가짜 사용자 데이터를 CSV로 생성"""
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        # CSV 헤더
        writer.writerow(["name", "email", "city", "age"])

        for _ in range(count):
            writer.writerow([
                fake.name(),
                fake.unique.email(),
                fake.city(),
                fake.random_int(min=20, max=60)
            ])

    print(f"[완료] CSV 생성: {filename} ({count}명)")


# ----------------------------------------
# 2. SQLite 테이블 생성 및 CSV 저장
# ----------------------------------------
def import_csv_to_db():
    """CSV 데이터를 SQLite에 저장"""
    with sqlite3.connect(DB_FILE) as conn:
        cur = conn.cursor()

        # 테이블 생성
        cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                city TEXT,
                age INTEGER
            )
        """)

        # CSV 읽기
        with open(CSV_FILE, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)

            cur.executemany("""
                INSERT INTO users (name, email, city, age)
                VALUES (:name, :email, :city, :age)
            """, [
                {
                    "name": row["name"],
                    "email": row["email"],
                    "city": row["city"],
                    "age": int(row["age"])
                }
                for row in reader
            ])

    print("[완료] SQLite 데이터 저장")


# ----------------------------------------
# 3. CREATE: 데이터 추가
# ----------------------------------------
def create_user(name, email, city, age):
    with sqlite3.connect(DB_FILE) as conn:
        cur = conn.cursor()

        cur.execute("""
            INSERT INTO users (name, email, city, age)
            VALUES (?, ?, ?, ?)
        """, (name, email, city, age))

        print(f"[CREATE] 추가된 사용자 ID: {cur.lastrowid}")


# ----------------------------------------
# 4. READ: 데이터 조회
# ----------------------------------------
def read_users():
    with sqlite3.connect(DB_FILE) as conn:
        cur = conn.cursor()

        cur.execute("""
            SELECT id, name, email, city, age
            FROM users
            ORDER BY id
            LIMIT 10
        """)

        rows = cur.fetchall()

        print("\n[READ] 사용자 목록")
        for row in rows:
            print(row)


def read_user_by_id(user_id):
    with sqlite3.connect(DB_FILE) as conn:
        cur = conn.cursor()

        cur.execute(
            "SELECT * FROM users WHERE id = ?",
            (user_id,)
        )

        print("\n[READ] 사용자 상세:", cur.fetchone())


# ----------------------------------------
# 5. UPDATE: 데이터 수정
# ----------------------------------------
def update_user(user_id, new_city):
    with sqlite3.connect(DB_FILE) as conn:
        cur = conn.cursor()

        cur.execute("""
            UPDATE users
            SET city = ?
            WHERE id = ?
        """, (new_city, user_id))

        print(f"[UPDATE] 변경된 행 수: {cur.rowcount}")


# ----------------------------------------
# 6. DELETE: 데이터 삭제
# ----------------------------------------
def delete_user(user_id):
    with sqlite3.connect(DB_FILE) as conn:
        cur = conn.cursor()

        cur.execute(
            "DELETE FROM users WHERE id = ?",
            (user_id,)
        )

        print(f"[DELETE] 삭제된 행 수: {cur.rowcount}")


# ----------------------------------------
# 7. 전체 실습 실행
# ----------------------------------------
if __name__ == "__main__":
    # 초기 데이터 생성 및 DB 적재
    create_csv()
    import_csv_to_db()

    # READ
    read_users()
    read_user_by_id(1)

    # CREATE
    create_user(
        "홍길동",
        "hong@example.com",
        "서울특별시",
        30
    )

    # READ: 추가 결과 확인
    read_user_by_id(101)

    # UPDATE
    update_user(101, "부산광역시")
    read_user_by_id(101)

    # DELETE
    delete_user(101)
    read_user_by_id(101)

    print("\n[완료] CRUD 실습 종료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.7 MB/s eta 0:00:00
[완료] CSV 생성: users.csv (100명)
[완료] SQLite 데이터 저장

[READ] 사용자 목록
(1, '박미정', 'seongjunyeong@example.org', '화천군', 26)
(2, '배영자', 'hyeonjun24@example.net', '군포시', 54)
(3, '강상현', 'songseongmin@example.com', '천안시 서북구', 49)
(4, '김성민', 'ghong@example.com', '용인시', 47)
(5, '이아름', 'gimminseo@example.com', '의정부시', 26)
(6, '권영호', 'yeonghwancoe@example.org', '논산시', 27)
(7, '신준혁', 'hyeonugim@example.com', '청주시 청원구', 40)
(8, '주현준', 'hyejin86@example.net', '부천시 오정구', 52)
(9, '김건우', 'gwangsugim@example.com', '광명시', 31)
(10, '유성현', 'hyeonjuu@example.com', '오산시', 40)

[READ] 사용자 상세: (1, '박미정', 'seongjunyeong@example.org', '화천군', 26)
[CREATE] 추가된 사용자 ID: 101

[READ] 사용자 상세: (101, '홍길동', 'hong@example.com', '서울특별시', 30)
[UPDATE] 변경된 행 수: 1

[READ] 사용자 상세: (101, '홍길동', 'hong@example.com', '부산광역시', 30)
[DELETE] 삭제된 행 수: 1

[READ] 사용자 상세: None

[완료] CRUD 실습 종료


In [27]:
import sqlite3

# DB 연결
conn = sqlite3.connect("test.db")
cur = conn.cursor()

In [29]:
# CREATE: 테이블 생성
cur.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER
    )
""")


In [26]:
# INSERT: 데이터 추가
cur.execute(
    "INSERT INTO users (name, age) VALUES (?, ?)",
    ("홍길동", 30)
)

# READ: 데이터 조회
cur.execute("SELECT * FROM users")
print(cur.fetchall())

# UPDATE: 데이터 수정
cur.execute(
    "UPDATE users SET age = ? WHERE name = ?",
    (31, "홍길동")
)

# DELETE: 데이터 삭제
cur.execute(
    "DELETE FROM users WHERE name = ?",
    ("홍길동",)
)

# 저장 및 연결 종료
conn.commit()
conn.close()

[(1, '홍길동', 30)]


In [30]:
import sqlite3

# DB 연결 및 자동 트랜잭션 처리
with sqlite3.connect("test.db") as conn:
    cur = conn.cursor()

    # CREATE: 테이블 생성
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT,
            age INTEGER
        )
    """)

    # INSERT: 데이터 추가
    cur.execute(
        "INSERT INTO users (name, age) VALUES (?, ?)",
        ("홍길동", 30)
    )

    # READ: 데이터 조회
    cur.execute("SELECT * FROM users")
    print(cur.fetchall())

    # UPDATE: 데이터 수정
    cur.execute(
        "UPDATE users SET age = ? WHERE name = ?",
        (31, "홍길동")
    )

    # DELETE: 데이터 삭제
    cur.execute(
        "DELETE FROM users WHERE name = ?",
        ("홍길동",)
    )

# with 블록 종료 시 트랜잭션 자동 처리
# 단, sqlite3.Connection의 with 문은 연결을 자동으로 닫지는 않음

[(1, '홍길동', 30)]


In [33]:
import sqlite3
from faker import Faker

fake = Faker("ko_KR")

with sqlite3.connect("test.db") as conn:
    cur = conn.cursor()

    # 기존 테이블이 존재하면 삭제하여 스키마 초기화
    cur.execute("DROP TABLE IF EXISTS users")

    # 테이블 다시 생성 (새로운 컬럼 스키마 반영)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT,
            email TEXT,
            city TEXT,
            age INTEGER
        )
    """)

    # 기존 데이터 삭제 후 100개 생성
    cur.execute("DELETE FROM users")

    # 데이터 100개 생성 및 저장
    users = [
        (
            fake.name(),
            fake.unique.email(),
            fake.city(),
            fake.random_int(min=20, max=60)
        )
        for _ in range(100)
    ]

    cur.executemany("""
        INSERT INTO users (name, email, city, age)
        VALUES (?, ?, ?, ?)
    """, users)

    # 데이터 조회
    cur.execute("SELECT * FROM users LIMIT 10")

    for row in cur.fetchall():
        print(row)

    # 전체 데이터 개수
    cur.execute("SELECT COUNT(*) FROM users")
    print("전체 데이터:", cur.fetchone()[0], "개")

(1, '김민석', 'songhayun@example.net', '고성군', 32)
(2, '강하윤', 'misuggweon@example.net', '공주시', 58)
(3, '이예준', 'boram64@example.net', '평택시', 28)
(4, '김서영', 'snam@example.net', '서천군', 48)
(5, '김지현', 'minji21@example.org', '용인시', 26)
(6, '박준서', 'rgim@example.net', '철원군', 45)
(7, '이경자', 'sugja23@example.net', '안양시', 45)
(8, '김은서', 'ean@example.com', '양구군', 53)
(9, '이정웅', 'ansunja@example.com', '철원군', 42)
(10, '진정숙', 'seojunsong@example.net', '평창군', 35)
전체 데이터: 100 개


In [37]:
import requests

In [39]:
requests.get("https://api.github.com/events")

<Response [200]>

In [38]:
r = requests.get('https://api.github.com/events')

In [40]:
r

<Response [200]>

In [42]:
r.content

b'[{"id":"24147785090","type":"PushEvent","actor":{"id":67440536,"login":"bryanhenrry94","display_login":"bryanhenrry94","gravatar_id":"","url":"https://api.github.com/users/bryanhenrry94","avatar_url":"https://avatars.githubusercontent.com/u/67440536?"},"repo":{"id":1078788356,"name":"bryanhenrry94/centraalinning","url":"https://api.github.com/repos/bryanhenrry94/centraalinning"},"payload":{"repository_id":1078788356,"push_id":46614350314,"ref":"refs/heads/main","head":"affba6fde8c972375974185e6db4a9b8d8471a56","before":"441e27b31caecf4602b970e5410b9facfdd3b99c"},"public":true,"created_at":"2026-10-15T15:45:42Z"},{"id":"24147784686","type":"PushEvent","actor":{"id":149322474,"login":"Ivanweb1","display_login":"Ivanweb1","gravatar_id":"","url":"https://api.github.com/users/Ivanweb1","avatar_url":"https://avatars.githubusercontent.com/u/149322474?"},"repo":{"id":1358212078,"name":"Ivanweb1/coupon-prototype","url":"https://api.github.com/repos/Ivanweb1/coupon-prototype"},"payload":{"repo

In [43]:
import json
import pandas as pd

# r-content가 bytes 형태인 경우
data = json.loads(r.content)

# 중첩 JSON을 평탄화하여 DataFrame 생성
df = pd.json_normalize(data)

# 주요 컬럼만 추출
df = df[[
    "id",
    "type",
    "actor.login",
    "repo.name",
    "payload.ref",
    "payload.head",
    "payload.before",
    "created_at",
    "public"
]]

# 컬럼명 변경
df.columns = [
    "event_id",
    "event_type",
    "username",
    "repository",
    "branch",
    "commit_sha",
    "previous_sha",
    "created_at",
    "public"
]

# 시간 변환 (UTC 기준)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)

print(df.head())

      event_id   event_type       username                    repository  \
0  24147785090    PushEvent  bryanhenrry94  bryanhenrry94/centraalinning   
1  24147784686    PushEvent       Ivanweb1     Ivanweb1/coupon-prototype   
2  24147784559    PushEvent  yongshuai1013       yongshuai1013/EiliStore   
3  24147784560  CreateEvent        afraghe                 afraghe/wfqdw   
4  24147784565    PushEvent   wistone-code        wistone-code/Fanikisha   

                                       branch  \
0                             refs/heads/main   
1  refs/heads/claude/friendly-thompson-ib4v5s   
2                             refs/heads/main   
3                                        main   
4                             refs/heads/main   

                                 commit_sha  \
0  affba6fde8c972375974185e6db4a9b8d8471a56   
1  79fd84c7fb9a7f79fa7bd8b32432f70682f8599d   
2  11cf1a19f3ce80886fe70b022dca606368bc6b5b   
3                                       NaN   
4  8572cb294e